In [1]:
import os
os.chdir('../')

In [2]:
import numpy as np

In [3]:
!ls

configs  data  models  nli_xy  notebooks  rep000


In [4]:
from nli_xy.tasks import load_tokenizer, parse_config, build_split_datasets, load_encoder_model, get_reps
import pandas as pd
import torch
DATA_DIR = './data/nlixy_small/'
CONF_FILE = './configs/compare_layers_conf.json'


In [5]:
config = parse_config.run(CONF_FILE)
config['device'] = 'cuda'
tokenizer = load_tokenizer.run(config)
encoder_model = load_encoder_model.run(config)

In [5]:
datasets = build_split_datasets.run(DATA_DIR, config, tokenizer)
rep_batches, meta_batches = get_reps.run(datasets['train'], encoder_model, config)

batches = zip(rep_batches, meta_batches)
flat_batches = []

100%|██████████| 46/46 [02:16<00:00,  2.98s/it]


In [11]:
last_layer = rep_batches[0][24]

In [12]:
meta_batch = meta_batches[0]

In [13]:
len(list(zip(meta_batch['X_range'][0], meta_batch['X_range'][1])))

64

In [14]:
import plotly.express as px
from sklearn.decomposition import PCA

In [15]:
def restructure(batch):
    rep_batch, meta_batch = batch
    batch_length = rep_batch[0].shape[0] # get one layer and its batch size
    num_tokens = rep_batch[0].shape[1] # get one layer and its batch size
    expanded_meta_batch = {
        'context': [],
        'layer_num': [],
        'token_num': [],
        'example_num': [], 
        'X_range': [],
    }

    expanded_rep_batch = []
    for layer_num, layer in enumerate(rep_batch):
        for token_num in range(num_tokens):
            expanded_meta_batch['X_range'] += list(zip(meta_batch['X_range'][0], meta_batch['X_range'][1]))
            for example_num in range(batch_length):

                expanded_meta_batch['context'].append(meta_batch['context'][example_num])
                expanded_meta_batch['layer_num']+=[layer_num]
                expanded_meta_batch['token_num']+=[token_num]
                expanded_rep_batch.append(layer[example_num, token_num, :].reshape(-1, 1024))

    expanded_rep_batch = torch.cat(expanded_rep_batch)
    return expanded_rep_batch, expanded_meta_batch

In [16]:
len(list(zip(meta_batch['X_range'][0], meta_batch['X_range'][1])))

64

In [17]:
batches = zip(rep_batches, meta_batches)
a, b = restructure(next(batches))

In [18]:
X = a
meta_df = b

In [19]:
pca = PCA(n_components=3)
pca.fit(X)
X_trans = pca.transform(X)

In [20]:
X_trans.shape

(160000, 3)

In [21]:
meta_df.keys()

dict_keys(['context', 'layer_num', 'token_num', 'example_num', 'X_range'])

In [22]:
df = pd.DataFrame(X_trans, columns=['x', 'y', 'z'])
df['layer_num'] = meta_df['layer_num']
df['context'] = meta_df['context']
df['token_num'] = meta_df['token_num']

In [23]:
!ls

configs  data  hyper  models  nli_xy  notebooks  rep000


In [208]:
fig = px.scatter_3d(df, x=df.x, y=df.y, z=df.z, color='layer_num')
fig.show()

ValueError: Value of 'color' is not the name of a column in 'data_frame'. Expected one of ['x', 'y', 'z', 'context', 'context_monotonicity'] but received: layer_num

# For Saved Embeddings:

In [211]:
# Load Example
pca = PCA(n_components=3)
pca.fit(X)
og_df = pd.read_csv('data/figure_data/test/embeddings.tsv', sep='\t')
X = og_df.to_numpy()
meta_df = pd.read_csv('data/figure_data/test/meta_df.tsv', sep='\t')
meta_df.columns

Index(['Unnamed: 0', 'premise', 'hypothesis', 'insertion_pair',
       'insertion_rel', 'context_monotonicity', 'context',
       'model_predictions'],
      dtype='object')

In [212]:
X_trans = pca.transform(X)
df = pd.DataFrame(X_trans, columns=['x', 'y', 'z'])
df['context'] = meta_df['context']
df['context_monotonicity'] = meta_df['context_monotonicity']

In [279]:
fig = px.scatter_3d(df, x=df.x, y=df.y, z=df.z, color='context_monotonicity')
fig.show()

In [273]:
import umap.umap_ as umap
X_umap = umap.UMAP(n_components=3).fit_transform(X)


In [274]:
X_umap.shape

(5355, 3)

In [ ]:
# Hellinger Distance

In [282]:
umap_df = pd.DataFrame(X_umap, columns=['x', 'y', 'z'])
umap_df['monotonicity'] = meta_df['context_monotonicity']
fig = px.scatter_3d(umap_df, x=umap_df.x, y=umap_df.y, z=umap_df.z, color='monotonicity')
fig.show()

In [72]:
from nli_xy.reach_probes.hyperrectangles import convex_hull, hull_volume, hull_intersect, relative_separation

In [73]:
!ls data/figure_data/test/

control_labels.tsv  labels.tsv	 monotonicity_control_labels.tsv
embeddings.tsv	    meta_df.tsv  monotonicity_labels.tsv


In [238]:
points_a = og_df.loc[df.context_monotonicity=='up'].to_numpy()
points_b = og_df.loc[df.context_monotonicity=='down'].to_numpy()
hull_a = convex_hull(points_a)
hull_b = convex_hull(points_b)

In [284]:
hull_a.shape

(1024, 2)

In [239]:
projected_a = pca.transform(points_a)
projected_b = pca.transform(points_b)

# Sample Hull Corners

In [240]:
def generate_edgepoints(num_points, hull):
    edge_points = []
    for point in range(num_points):
        selection = np.random.randint(0,2, 1024)
        edge_point = np.zeros(1024)
        for dim in range(1024):
            edge_point[dim] = hull[dim, selection[dim]]
        edge_points.append(edge_point)
    return edge_points

In [285]:
sample_hull_a = np.stack(generate_edgepoints(2000, hull_a))
sample_hull_b = np.stack(generate_edgepoints(2000, hull_b))

In [286]:
sample_hull_a.shape

(2000, 1024)

In [287]:
# for each dimension, we've chosen either min or max
#sample_hull_a[:,0]

In [288]:
proj_hull_a = pca.transform(sample_hull_a)
proj_hull_b = pca.transform(sample_hull_b)

In [289]:
import plotly.graph_objects as go

fig = px.scatter_3d(x=projected_a[:,0], y=projected_a[:,1], z=projected_a[:,2])

fig.add_trace(go.Scatter3d(x=proj_hull_a[:,0], y=proj_hull_a[:,1], z=proj_hull_a[:,2],mode='markers'))
fig.add_trace(go.Scatter3d(x=proj_hull_b[:,0], y=proj_hull_b[:,1], z=proj_hull_b[:,2],mode='markers'))

In [290]:
fig = px.scatter_3d(x=projected_b[:,0], y=projected_b[:,1], z=projected_b[:,2], color_discrete_sequence=['purple'])
fig.add_trace(go.Scatter3d(x=proj_hull_a[:,0], y=proj_hull_a[:,1], z=proj_hull_a[:,2],mode='markers'))
fig.add_trace(go.Scatter3d(x=proj_hull_b[:,0], y=proj_hull_b[:,1], z=proj_hull_b[:,2],mode='markers'))
fig.show()

In [247]:
#volumes
print(hull_volume(hull_a))
print(hull_volume(hull_b))

2.5492452186452588
2.417135369417613


In [230]:
hull_inter = hull_intersect(hull_a, hull_b)
print(hull_volume(hull_inter))

2.294147293793164
